In [ ]:
import torch
import json
from PIL import Image
from transformers import AutoModel, AutoTokenizer

In [2]:
def exact_match_score(predictions, keywords):
    score = 0.0
    keywords_copy = keywords.copy()
    
    for prediction in predictions:
        for keyword in keywords_copy:
            if keyword.lower() in prediction.lower():
                score += 1.0 / len(keywords)
                keywords_copy.remove(keyword)
    
    return score

In [3]:
def compute_exact_match_score(model, tokenizer, json_file, result_file):
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    total_score = 0.0
    
    for idx, entry in enumerate(data):
        entry_score = 0.0
        
        image_path = entry['image'].replace('../', '../../')
        image = Image.open(image_path).convert('RGB')
        keywords = entry['keywords']
        resp = []
        for i, conversation in enumerate(entry['conversations']):
            if conversation['role'] == 'user':
                question = conversation['content']
                if "<image>\\n" in question:
                    question = question.replace("<image>\\n", "")
                
                msgs = [
                    {
                        'role': 'user',
                        'content': question
                    }
                ]
                res, context, _ = model.chat(
                    image=image,
                    msgs=msgs,
                    context=None,
                    tokenizer=tokenizer,
                    sampling=True,
                    temperature=0.7,
                    max_new_tokens=128
                )
                print(res)
                resp.append(res)
                
                if i + 1 < len(entry['conversations']):
                    entry['conversations'][i + 1]['answer'] = res
                    
        entry_score += exact_match_score(resp, keywords)
        total_score += entry_score
        print(f"Entry {idx + 1} score: {entry_score}")
    with open(result_file, 'w') as f:
        json.dump(data, f, indent=4)
    print(f"Total exact match score: {total_score}")

In [4]:
model = AutoModel.from_pretrained('../../output/minicpmv_sensitive_fiofp_emmu', trust_remote_code=True, torch_dtype=torch.bfloat16)
model = model.to(device='cuda:1', dtype=torch.bfloat16)

tokenizer = AutoTokenizer.from_pretrained('../../output/minicpmv_sensitive_fiofp_emmu', trust_remote_code=True)
model.eval()

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.02it/s]


MiniCPMV(
  (llm): MiniCPMForCausalLM(
    (model): MiniCPMModel(
      (embed_tokens): Embedding(122753, 2304)
      (layers): ModuleList(
        (0-39): 40 x MiniCPMDecoderLayer(
          (self_attn): MiniCPMSdpaAttention(
            (q_proj): Linear(in_features=2304, out_features=2304, bias=False)
            (k_proj): Linear(in_features=2304, out_features=2304, bias=False)
            (v_proj): Linear(in_features=2304, out_features=2304, bias=False)
            (o_proj): Linear(in_features=2304, out_features=2304, bias=False)
            (rotary_emb): MiniCPMRotaryEmbedding()
          )
          (mlp): MiniCPMMLP(
            (gate_proj): Linear(in_features=2304, out_features=5760, bias=False)
            (up_proj): Linear(in_features=2304, out_features=5760, bias=False)
            (down_proj): Linear(in_features=5760, out_features=2304, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): MiniCPMRMSNorm()
          (post_attention_layernorm): Min

In [ ]:
sensitive_file = '../../data/vqa/FIOFP/FIOFP_100.json'
sensitive_result = '../../output/model_output/emmu_model/FIOFP_100_output.json'
compute_exact_match_score(model, tokenizer, sensitive_file, sensitive_result)